In [1]:
import pandas as pd
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [2]:
# Tải dataset ở Chế độ 'distractor'
print("Đang tải dữ liệu từ Hugging Face...")
dataset = load_dataset("hotpotqa/hotpot_qa", "distractor", split="train")

# Chuyển sang Pandas DataFrame để xử lý
df = dataset.to_pandas()
print(f"Tổng số mẫu ban đầu: {len(df)}")

Đang tải dữ liệu từ Hugging Face...
Tổng số mẫu ban đầu: 90447


In [3]:
# FEATURE ENGINEERING

def classify_answer_type(answer):
    """Phân loại: Là câu hỏi Yes/No hay câu hỏi trích xuất (Span)"""
    if str(answer).lower() in ['yes', 'no']:
        return 'yes_no'
    return 'span'

def count_supporting_facts(supp_facts):
    """Đếm số lượng supporting facts cần thiết"""
    # Trong HF dataset, supporting_facts là dict: {'title': [], 'sent_id': []}
    count = len(supp_facts['title'])
    # Gom nhóm các trường hợp hiếm (ví dụ > 4 facts) để tránh lỗi khi lấy mẫu
    return str(count) if count <= 4 else '5+'

df['answer_type'] = df['answer'].apply(classify_answer_type)
df['num_facts'] = df['supporting_facts'].apply(count_supporting_facts)

# "Stratify Key" - Chìa khóa phân tầng tổng hợp
# Example: "hard_span_2" (Câu khó, trả lời trích xuất, cần 2 facts)
df['stratify_key'] = (df['level'] + "_" + df['answer_type'] + "_" + df['num_facts'])

In [4]:
# SAMPLING

TARGET_SIZE = 1000

# Loại bỏ các nhóm quá nhỏ không thể chia tập train/test được
group_counts = df['stratify_key'].value_counts()
valid_groups = group_counts[group_counts >= 2].index
df_filtered = df[df['stratify_key'].isin(valid_groups)]

# Stratified Sampling
try:
    _, df_sampled = train_test_split(
        df_filtered,
        test_size=TARGET_SIZE,
        stratify=df_filtered['stratify_key'],
        random_state=42 # Cố định seed để kết quả tái lập được
    )
    print(f"\nĐã lấy mẫu thành công: {len(df_sampled)} mẫu.")
except ValueError as e:
    print(f"Lỗi phân tầng: {e}. Đang chuyển sang lấy mẫu ngẫu nhiên (Random Sampling)...")
    df_sampled = df.sample(n=TARGET_SIZE, random_state=42)


Đã lấy mẫu thành công: 1000 mẫu.


In [5]:
# So sánh tỷ lệ giữa Tập Gốc và Tập Mới

def compare_distributions(original, sampled, column, name):
    orig_dist = original[column].value_counts(normalize=True)
    samp_dist = sampled[column].value_counts(normalize=True)
    
    comp_df = pd.DataFrame({
        'Gốc (%)': orig_dist * 100,
        'Mới (%)': samp_dist * 100
    }).sort_index()
    
    print(f"\n--- Phân phối theo {name} ---")
    print(comp_df.round(2))

# In kết quả so sánh
compare_distributions(df, df_sampled, 'level', 'Độ khó (Level)')
compare_distributions(df, df_sampled, 'answer_type', 'Loại câu trả lời (Yes/No vs Span)')
compare_distributions(df, df_sampled, 'num_facts', 'Số lượng Supporting Facts')


--- Phân phối theo Độ khó (Level) ---
        Gốc (%)  Mới (%)
level                   
easy      19.87     19.8
hard      17.32     17.3
medium    62.81     62.9

--- Phân phối theo Loại câu trả lời (Yes/No vs Span) ---
             Gốc (%)  Mới (%)
answer_type                  
span           93.94     94.0
yes_no          6.06      6.0

--- Phân phối theo Số lượng Supporting Facts ---
           Gốc (%)  Mới (%)
num_facts                  
2            70.40     70.4
3            22.13     22.1
4             6.43      6.4
5+            1.04      1.1


In [6]:
# XUẤT DỮ LIỆU ĐỂ SỬ DỤNG

# Xóa các cột tạm để trả về định dạng gốc
final_columns = dataset.column_names
df_final = df_sampled[final_columns]

# Chuyển ngược lại thành HuggingFace Dataset
mini_dataset = Dataset.from_pandas(df_final, preserve_index=False)

# Lưu xuống ổ cứng
mini_dataset.save_to_disk("hotpot_mini_1k")
print("Ví dụ mẫu đầu tiên trong tập mới:")
print(mini_dataset[0]['question'])
print("Level:", mini_dataset[0]['level'])

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Ví dụ mẫu đầu tiên trong tập mới:
Which author was also an activist, Jonathan Kellerman or Alan Paton? 
Level: medium
